Module 5 Assignment 

In [6]:
import sys, time, re, json, pathlib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)
GENES = ["CYP2C19", "CYP2C9", "VKORC1", "DPYD", "TPMT"]
MV = "https://myvariant.info/v1"
AVI_FILE = DATA / "alphagenome_avi.tsv"

ANCHORS = {"rs1799853": "CYP2C9*2", "rs1057910": "CYP2C9*3",
           "rs9923231": "VKORC1 -1639G>A", "rs4244285": "CYP2C19*2",
           "rs12248560": "CYP2C19*17", "rs3918290": "DPYD*2A",
           "rs67376798": "DPYD c.2846A>T", "rs1142345": "TPMT*3C"}

FIELDS = ",".join([
    "dbsnp.rsid",                        # the rs number
    "snpeff.ann.effect",                 # consequence term
    "cadd.phred",                        # CADD
    "dbnsfp.revel.score",                # REVEL
    "dbnsfp.alphamissense",              # AlphaMissense
    "dbnsfp.aa.ref", "dbnsfp.aa.alt", "dbnsfp.aa.pos", "dbnsfp.genename",
    "gnomad_genome.af.af", "gnomad_exome.af.af",
    "clinvar.rcv.clinical_significance",
    "clinvar.rcv.review_status",
    "clinvar.gene.symbol",
    "clinvar.variant_id",                # join key for CADD (see Section 3)
])

def first(x):
    """Some fields come back as a single value, some as a list. Take the first."""
    return (x[0] if x else None) if isinstance(x, list) else x

def dig(d, *path):
    """Walk a nested dict safely. Returns None if any level is missing."""
    for k in path:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
        if d is None:
            return None
    return d

# ClinVar review status, as the familiar star rating
STARS = {"practice guideline": 4,
         "reviewed by expert panel": 3,
         "criteria provided, multiple submitters, no conflicts": 2,
         "criteria provided, conflicting classifications": 1,
         "criteria provided, conflicting interpretations": 1,
         "criteria provided, single submitter": 1,
         "no assertion criteria provided": 0,
         "no classification provided": 0}

# snpEff reports one effect per transcript. Keep the most severe one.
SEVERITY = ["stop_gained", "frameshift_variant", "splice_acceptor_variant",
            "splice_donor_variant", "start_lost", "stop_lost", "missense_variant",
            "splice_region_variant", "synonymous_variant", "5_prime_UTR_variant",
            "3_prime_UTR_variant", "intron_variant", "upstream_gene_variant",
            "downstream_gene_variant", "intergenic_region"]

def all_effects(h):
    """Every effect snpEff reported, across every transcript, split into single terms."""
    ann = dig(h, "snpeff", "ann")
    raw = [a.get("effect") for a in (ann if isinstance(ann, list) else [ann]) if isinstance(a, dict)]
    terms = []
    for e in raw:                       # snpEff emits compound terms joined by "&"
        if e:
            terms += [t for t in e.split("&") if t != "structural_interaction_variant"]
    return sorted(set(terms))

def worst_effect(h):
    """Collapse to ONE term: the most severe across all transcripts."""
    terms = all_effects(h)
    if not terms:
        return None
    return min(terms, key=lambda t: SEVERITY.index(t) if t in SEVERITY else 99)

def to_row(h, gene=None):
    aaref, aaalt = first(dig(h, "dbnsfp", "aa", "ref")), first(dig(h, "dbnsfp", "aa", "alt"))
    aapos = first(dig(h, "dbnsfp", "aa", "pos"))
    rcv = first(dig(h, "clinvar", "rcv")) or {}   # one record per submission; take the first
    rev = rcv.get("review_status")
    am = dig(h, "dbnsfp", "alphamissense", "score")
    return {
        "hgvs_g":        h.get("_id"),
        "rsid":          dig(h, "dbsnp", "rsid"),
        "gene":          first(dig(h, "clinvar", "gene", "symbol")) or first(dig(h, "dbnsfp", "genename")) or gene,
        "clinvar_id":    dig(h, "clinvar", "variant_id"),
        "consequence":   worst_effect(h),
        "consequence_all": ";".join(all_effects(h)),   # keep the full picture
        "aa_change":     f"{aaref}{aapos}{aaalt}" if (aaref and aaalt and aapos) else None,
        "cadd_phred":    dig(h, "cadd", "phred"),
        "revel":         first(dig(h, "dbnsfp", "revel", "score")),
        "alphamissense": first(am) if isinstance(am, list) else am,
        "gnomad_af":     dig(h, "gnomad_genome", "af", "af") or dig(h, "gnomad_exome", "af", "af"),
        "clinvar_sig":   rcv.get("clinical_significance"),
        "clinvar_stars": STARS.get(str(rev).lower(), 0),
    }

FORCE_REFETCH = False

# Known pharmacogenomic alleles, so we can check the fetch worked and
# recognise them later.
ANCHORS = {"rs1799853": "CYP2C9*2", "rs1057910": "CYP2C9*3",
           "rs9923231": "VKORC1 -1639G>A", "rs4244285": "CYP2C19*2",
           "rs12248560": "CYP2C19*17", "rs3918290": "DPYD*2A",
           "rs67376798": "DPYD c.2846A>T", "rs1142345": "TPMT*3C"}

def fetch_gene(gene, verbose=True):
    """Search ClinVar for one gene, annotate every hit, return a tidy DataFrame."""
    r = requests.get(f"{MV}/query", timeout=90, params={
        "q": f"clinvar.gene.symbol:{gene}", "assembly": "hg38",
        "fields": "_id", "size": 1000})
    r.raise_for_status()
    ids = [h["_id"] for h in r.json().get("hits", [])]
    if verbose:
        print(f"  {gene:9s} {len(ids):>5} variants found", end="", flush=True)

    rows = []
    for i in range(0, len(ids), 500):
        rr = requests.post(f"{MV}/variant", timeout=180,
                           data={"ids": ",".join(ids[i:i+500]),
                                 "fields": FIELDS, "assembly": "hg38"})
        rr.raise_for_status()
        rows += [to_row(h, gene) for h in rr.json()]
        time.sleep(0.3)                        # be polite to a free API

    df = pd.DataFrame(rows).drop_duplicates("hgvs_g")

    # MyVariant.info only carries genome-wide CADD on hg19. Ask the same
    # question on hg19 and join on the ClinVar variant ID, which is build-free.
    r19 = requests.get(f"{MV}/query", timeout=90, params={
        "q": f"clinvar.gene.symbol:{gene}", "assembly": "hg19",
        "fields": "clinvar.variant_id,cadd.phred", "size": 1000})
    r19.raise_for_status()
    hits19 = r19.json().get("hits", [])
    cadd = {}
    for h in hits19:
        vid = first(dig(h, "clinvar", "variant_id"))      # occasionally a list
        ph = dig(h, "cadd", "phred")
        if vid is not None and ph is not None:
            cadd[vid] = ph
    df["cadd_phred"] = df.clinvar_id.map(cadd)

    if len(ids) >= 1000 or len(hits19) >= 1000:
        print(f"\n  [!] {gene} hit the 1000-result cap. The two builds may have been")
        print( "      truncated to different sets, so some CADD scores will be missing.")
    if verbose:
        print(f"  ->  {len(df)} annotated")
    return df

def numeric(df):
    """Force the score columns to numbers. Strings here break everything later."""
    for c in ["cadd_phred", "revel", "alphamissense", "spliceai_dg",
              "spliceai_dl", "avi", "gnomad_af", "clinvar_stars"]:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def clinvar_label(sig):
    if not isinstance(sig, str): return "none"
    s = sig.lower()
    if "conflict" in s: return "conflicting"
    if "pathogenic" in s and "non-pathogenic" not in s: return "P/LP"
    if "benign" in s: return "B/LB"
    if "uncertain" in s: return "VUS"
    return "other"          # includes "drug response"

STRENGTH = [(18.7, "strong, pathogenic"), (4.3, "moderate, pathogenic"),
            (2.08, "supporting, pathogenic"), (0.48, "no evidence either way"),
            (0.23, "supporting, benign"), (0.053, "moderate, benign"),
            (0.0, "strong, benign")]

def strength_for(lr):
    if lr != lr: return "undefined"
    for cut, name in STRENGTH:
        if lr >= cut: return name
    return "strong, benign"

def calibrate(df, score_col, bands):
    d = df[df.label.isin(["P/LP", "B/LB"]) & df[score_col].notna()].copy()
    d["band"] = pd.cut(d[score_col], bands)
    n_p, n_b = (d.label == "P/LP").sum(), (d.label == "B/LB").sum()
    g = d.groupby("band", observed=False)
    out = pd.DataFrame({"n": g.size(),
                        "n_PLP": g.label.apply(lambda s: (s == "P/LP").sum()),
                        "n_BLB": g.label.apply(lambda s: (s == "B/LB").sum())})
    with np.errstate(divide="ignore", invalid="ignore"):
        out["LR+"] = ((out.n_PLP / max(n_p, 1)) / (out.n_BLB / max(n_b, 1))).round(2)
    out["evidence"] = out["LR+"].map(strength_for)
    out.attrs["n_p"], out.attrs["n_b"] = n_p, n_b
    return out

print("helpers loaded:", ", ".join(["fetch_gene", "numeric", "to_row", "clinvar_label",
                                    "calibrate", "strength_for", "worst_effect"]))

helpers loaded: fetch_gene, numeric, to_row, clinvar_label, calibrate, strength_for, worst_effect


In [7]:
MY_GENE = "CYP2C9"        # <-- set this to YOUR assigned gene
assert MY_GENE in GENES, f"MY_GENE must be one of {GENES}"
MY_FILE = DATA / f"{MY_GENE}.tsv"
print("working on", MY_GENE)

working on CYP2C9


In [8]:
# TODO Q1
#   (a) if MY_FILE exists, read it; otherwise fetch_gene(MY_GENE) and save it
#   (b) pass it through numeric() so scores are numbers, not strings
#   (c) add a `label` column using clinvar_label()
#   (d) run and print the three quality checks described above
#
# Everything you need is already defined. This is about eight lines.

if MY_FILE.exists():
    G = pd.read_csv(MY_FILE, sep="\t")
else:
    G = fetch_gene(MY_GENE)
    G.to_csv(MY_FILE, sep="\t", index=False)

G = numeric(G)
G["label"] = G["clinvar_sig"].apply(clinvar_label)

print(f"\n{MY_GENE}: {len(G)} variants fetched\n")

print("Anchor alleles found:")
anchors_in_gene = {rsid: name for rsid, name in ANCHORS.items() if MY_GENE in name}
for rsid, name in anchors_in_gene.items():
    found = rsid in G["rsid"].values
    print(f"  {name} ({rsid}): {'FOUND' if found else 'MISSING'}")

print("\nScore column coverage:")
for col in ["cadd_phred", "revel", "alphamissense", "gnomad_af"]:
    n = G[col].notna().sum()
    print(f"  {col}: {n} / {len(G)} populated ({100*n/len(G):.1f}%)")


CYP2C9: 41 variants fetched

Anchor alleles found:
  CYP2C9*2 (rs1799853): FOUND
  CYP2C9*3 (rs1057910): FOUND

Score column coverage:
  cadd_phred: 38 / 41 populated (92.7%)
  revel: 33 / 41 populated (80.5%)
  alphamissense: 33 / 41 populated (80.5%)
  gnomad_af: 29 / 41 populated (70.7%)


In [9]:
# TODO Q2
#   (a) group G by consequence and count non-null scores per tool
#   (b) print the table, plus the overall percentage scored by each tool
#   (c) print which consequence classes have ZERO REVEL scores
#
# Hint: .groupby("consequence") then .apply(lambda s: s.notna().sum())

TOOLS = ["cadd_phred", "revel", "alphamissense"]

coverage = G.groupby("consequence")[TOOLS].apply(lambda d: d.notna().sum())
print("Coverage by consequence class:\n")
print(coverage.to_string())

print("\nOverall percentage scored by each tool:")
for t in TOOLS:
    pct = 100 * G[t].notna().sum() / len(G)
    print(f"  {t}: {pct:.1f}%")

zero_revel = coverage[coverage["revel"] == 0].index.tolist()
print(f"\nConsequence classes with ZERO REVEL scores: {zero_revel}")

Coverage by consequence class:

                    cadd_phred  revel  alphamissense
consequence                                         
frameshift_variant           0      0              0
intron_variant               2      0              0
missense_variant            33     33             33
synonymous_variant           3      0              0

Overall percentage scored by each tool:
  cadd_phred: 92.7%
  revel: 80.5%
  alphamissense: 80.5%

Consequence classes with ZERO REVEL scores: ['frameshift_variant', 'intron_variant', 'synonymous_variant']


In [10]:
print(G[G["rsid"].isin(["rs1799853", "rs1057910"])][["rsid", "gnomad_af", "clinvar_sig"]])

         rsid  gnomad_af    clinvar_sig
7   rs1057910   0.049454  drug response
37  rs1799853   0.088164  drug response


In [11]:
# TODO Q3
#   (a) set MAX_AF to a threshold you can defend for YOUR gene
#   (b) report how many variants it removes
#   (c) CRITICAL: list the P/LP or "drug response" variants your filter removes.
#       If it removes a known functional allele, either change the threshold or
#       justify keeping it in your report.
#   (d) for contrast, report what a 0.001 rare-disease filter would have removed
#   (e) set G to the filtered set

MAX_AF = ...     # <- your choice

MAX_AF = 0.10

before = len(G)
removed = G[(G["gnomad_af"] > MAX_AF)]
print(f"MAX_AF = {MAX_AF}: removes {len(removed)} of {before} variants\n")

removed_important = removed[removed["label"].isin(["P/LP"]) | (removed["clinvar_sig"].str.contains("drug response", na=False))]
print("P/LP or drug-response variants removed by this filter:")
print(removed_important[["rsid", "gnomad_af", "clinvar_sig"]].to_string(index=False) if len(removed_important) else "  (none)")

contrast = G[G["gnomad_af"] > 0.001]
contrast_important = contrast[contrast["label"].isin(["P/LP"]) | (contrast["clinvar_sig"].str.contains("drug response", na=False))]
print(f"\nFor contrast, a 0.001 rare-disease filter would have removed {len(contrast)} variants,")
print("including these P/LP or drug-response ones:")
print(contrast_important[["rsid", "gnomad_af", "clinvar_sig"]].to_string(index=False) if len(contrast_important) else "  (none)")

G = G[(G["gnomad_af"].isna()) | (G["gnomad_af"] <= MAX_AF)]
print(f"\nG now has {len(G)} variants after filtering")

MAX_AF = 0.1: removes 2 of 41 variants

P/LP or drug-response variants removed by this filter:
  (none)

For contrast, a 0.001 rare-disease filter would have removed 10 variants,
including these P/LP or drug-response ones:
      rsid  gnomad_af   clinvar_sig
rs28371685   0.007015 drug response
 rs1057910   0.049454 drug response
 rs7900194   0.015809 drug response
 rs1799853   0.088164 drug response

G now has 39 variants after filtering


In [12]:
# TODO Q4
#   (a) choose SCORE_COL and BANDS and run calibrate() on G
#   (b) print how many labelled variants the calibration rests on
#   (c) print how many bands rest on fewer than 10 variants
#   (d) if your gene is too small to calibrate, say so in the output and take
#       one of the two documented routes above
#
# Suggested bands: CADD [0,10,20,25,30,50]   REVEL [0,.2,.4,.6,.8,1.0]

SCORE_COL = ...   # <- your choice
BANDS     = ...   # <- your bin edge

SCORE_COL = "cadd_phred"
BANDS = [0, 10, 20, 25, 30, 50]

result = calibrate(G, SCORE_COL, BANDS)
print(f"Calibrating {SCORE_COL} on {MY_GENE}\n")
print(result.to_string())

n_p, n_b = result.attrs["n_p"], result.attrs["n_b"]
print(f"\nThis calibration rests on {n_p} P/LP and {n_b} B/LB labelled variants ({n_p + n_b} total)")

small_bands = result[result["n"] < 10]
print(f"\nBands resting on fewer than 10 variants: {len(small_bands)}")
if len(small_bands):
    print(small_bands.to_string())

Calibrating cadd_phred on CYP2C9

          n  n_PLP  n_BLB   LR+                evidence
band                                                   
(0, 10]   4      1      3  2.33  supporting, pathogenic
(10, 20]  1      0      1  0.00          strong, benign
(20, 25]  3      0      3  0.00          strong, benign
(25, 30]  0      0      0   NaN               undefined
(30, 50]  0      0      0   NaN               undefined

This calibration rests on 1 P/LP and 7 B/LB labelled variants (8 total)

Bands resting on fewer than 10 variants: 5
          n  n_PLP  n_BLB   LR+                evidence
band                                                   
(0, 10]   4      1      3  2.33  supporting, pathogenic
(10, 20]  1      0      1  0.00          strong, benign
(20, 25]  3      0      3  0.00          strong, benign
(25, 30]  0      0      0   NaN               undefined
(30, 50]  0      0      0   NaN               undefined


In [13]:
print(f"\nCAUTION: all {len(result)} bands rest on fewer than 10 variants (max band n={result['n'].max()}).")
print("This calibration should be treated as illustrative only, not reliable evidence.")
print("The apparent reversal (low CADD -> LR 2.33, higher CADD -> LR 0.00) is most consistent")
print("with small-sample noise rather than a real biological signal.")


CAUTION: all 5 bands rest on fewer than 10 variants (max band n=4).
This calibration should be treated as illustrative only, not reliable evidence.
The apparent reversal (low CADD -> LR 2.33, higher CADD -> LR 0.00) is most consistent
with small-sample noise rather than a real biological signal.


In [14]:
# TODO Q5
#   (a) take the VUS rows from G
#   (b) map each one to the evidence strength of its calibrated band
#   (c) count how many move towards pathogenic, towards benign, and stay put
#   (d) export to pgx_classified_<MY_GENE>.csv with these columns:
#       rsid, gene, consequence, aa_change, your score column, gnomad_af,
#       evidence, provisional_call

vus = G[G["label"] == "VUS"].copy()
print(f"{len(vus)} VUS variants in {MY_GENE}\n")

vus["band"] = pd.cut(vus[SCORE_COL], BANDS)
band_evidence = result["evidence"].to_dict()
vus["evidence"] = vus["band"].map(band_evidence)
vus["evidence"] = vus["evidence"].fillna("undefined, not enough data")

def provisional_call(ev):
    if "pathogenic" in str(ev) and "no evidence" not in str(ev):
        return "leans pathogenic"
    elif "benign" in str(ev):
        return "leans benign"
    else:
        return "stays VUS"

vus["provisional_call"] = vus["evidence"].apply(provisional_call)

print(vus["provisional_call"].value_counts().to_string())

out = vus[["rsid", "gene", "consequence", "aa_change", SCORE_COL, "gnomad_af", "evidence", "provisional_call"]].copy()
out = out.rename(columns={SCORE_COL: "score"})
out_file = f"pgx_classified_{MY_GENE}.csv"
out.to_csv(out_file, index=False)
print(f"\nWrote {out_file} ({len(out)} rows)")

22 VUS variants in CYP2C9

provisional_call
leans pathogenic    9
leans benign        9
stays VUS           4

Wrote pgx_classified_CYP2C9.csv (22 rows)


In [15]:
import os
print(os.environ.get('ALPHAGENOME_API_KEY'))

REDACTED


In [18]:
SKIP_ATLAS   = False
MAX_VARIANTS = 200
# ... (the rest of the cell, unchanged)

In [20]:
d = G.copy()
coding = d.consequence.isin(["missense_variant", "synonymous_variant"])

In [19]:
SKIP_ATLAS   = False
MAX_VARIANTS = 200

import os
ATLAS_STATUS = "not run"

def to_variant_string(hgvs):
    """Atlas wants 1-based GRCh38 chr:pos:ref>alt. It does NOT accept rsIDs."""
    m = re.match(r"(chr[\w]+):g\.(\d+)([ACGT])>([ACGT])", str(hgvs))
    return f"{m.group(1)}:{m.group(2)}:{m.group(3)}>{m.group(4)}" if m else None

if SKIP_ATLAS:
    print("SKIPPED. Disclose this in your report and answer Q6.2 from reasoning.")
    ATLAS_STATUS = "skipped"
else:
    try:
        from alphagenome.atlas import atlas
        from alphagenome.data import genome
        key = os.environ.get("ALPHAGENOME_API_KEY") or os.environ.get("ALPHA_GENOME_API_KEY")
        if not key:
            raise RuntimeError(
                "No API key in your environment. Did you relaunch Jupyter from a NEW\n"
                "  terminal after setting it? Check with:\n"
                "    import os; print(os.environ.get('ALPHAGENOME_API_KEY'))\n"
                "  Or set SKIP_ATLAS = True.")
        client = atlas.create(key, timeout=30)

        pri = G.assign(_nc=~G.consequence.fillna("").str.contains("missense")) \
               .sort_values("_nc", ascending=False).head(MAX_VARIANTS)
        rows, errs = [], 0
        for n, (_, r) in enumerate(pri.iterrows(), 1):
            vs = to_variant_string(r.hgvs_g)
            if not vs:
                continue
            try:
                res = client.query_variant(
                    genome.Variant.from_str(vs),
                    requested_scorers=["AVI_SCORE", "AVI_SCORE_FEATURE_IMPORTANCE"])
                ad = res["AVI_SCORE"]
                cdf = float(ad.layers["quantiles"][0, 0])
                fi = res["AVI_SCORE_FEATURE_IMPORTANCE"]
                rows.append({"rsid": r.rsid,
                             "avi_phred": round(-10*np.log10(max(1-cdf, 1e-12)), 2),
                             "top_feature": fi.var["name"].values[int(np.argmax(np.abs(fi.X[0])))]})
            except Exception:
                errs += 1
            if n % 50 == 0:
                print(f"  {n}/{len(pri)} queried", flush=True)

        avi = pd.DataFrame(rows)
        if len(avi):
            avi.to_csv(DATA / f"avi_{MY_GENE}.tsv", sep="\t", index=False)
            G = G.merge(avi, on="rsid", how="left")
            G["avi"] = G.avi_phred
            ATLAS_STATUS = "queried live"
            print(f"\nscored {len(avi)} variants ({errs} failures)")
    except ImportError:
        print('alphagenome not installed. pip install "alphagenome>=0.9.0", then restart.')
        print("Or set SKIP_ATLAS = True.")
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

if "avi" not in G.columns: G["avi"] = np.nan
if "top_feature" not in G.columns: G["top_feature"] = np.nan
G["avi"] = pd.to_numeric(G["avi"], errors="coerce")

if G.avi.notna().sum() > 0:
    a = G[G.avi.notna()]
    cols = [c for c in ["rsid", "consequence", "avi_phred", "top_feature",
                        "revel", "cadd_phred", "label"] if c in a.columns]
    print(f"\nhighest-impact {MY_GENE} variants by AVI\n")
    print(a.sort_values("avi", ascending=False)[cols].head(10).to_string(index=False))
    print("\nwhat is driving the scores (read this BEFORE the numbers)")
    print(a.top_feature.value_counts().head(8).to_string())
else:
    nc = G[~G.consequence.fillna("").str.contains("missense")]
    print(f"\nnon-missense variants in {MY_GENE}, which is what the Atlas is for: {len(nc)}")
    print(nc.consequence.value_counts().to_string())

print(f"\n[Atlas status: {ATLAS_STATUS}]")


scored 36 variants (2 failures)

highest-impact CYP2C9 variants by AVI

        rsid        consequence  avi_phred     top_feature  revel  cadd_phred label
 rs370407644   missense_variant      23.41 MERGED_SPLICING  0.144      23.300   VUS
rs1226372739   missense_variant      22.20   ALPHAMISSENSE  0.633      26.300   VUS
         NaN   missense_variant      22.03   ALPHAMISSENSE  0.146      25.300   VUS
         NaN   missense_variant      22.03   ALPHAMISSENSE  0.094      16.040 other
         NaN                NaN      22.03   ALPHAMISSENSE    NaN         NaN other
         NaN   missense_variant      22.03   ALPHAMISSENSE  0.123       0.002  B/LB
         NaN   missense_variant      22.03   ALPHAMISSENSE  0.236      23.600   VUS
         NaN   missense_variant      22.03   ALPHAMISSENSE  0.281      24.100   VUS
         NaN frameshift_variant      22.03   ALPHAMISSENSE    NaN         NaN  B/LB
         NaN   missense_variant      22.03   ALPHAMISSENSE  0.320      26.500   VUS

wh

In [21]:
d = G.copy()
coding = d.consequence.isin(["missense_variant", "synonymous_variant"])

d["flag_cadd_vs_revel"]  = (d.cadd_phred > 25) & (d.revel < 0.35)
d["flag_revel_vs_am"]    = (d.revel - d.alphamissense).abs() > 0.4
d["flag_scored_but_vus"] = (d.label == "VUS") & (d.revel > 0.7)
d["flag_common_but_plp"] = (d.gnomad_af > 0.01) & (d.label.isin(["P/LP", "other"]))
d["flag_no_tool_at_all"] = d[["revel", "alphamissense", "cadd_phred"]].isna().all(axis=1)
if d.avi.notna().any():
    d["flag_atlas_only"] = (d.avi > 20) & d.revel.isna() & d.alphamissense.isna()

flags = [c for c in d.columns if c.startswith("flag_")]
print(f"discordant variants in {MY_GENE}\n")
print(d[flags].sum().to_string())

hits = d[d[flags].any(axis=1)]
cols = [c for c in ["rsid", "note", "consequence", "consequence_all", "aa_change",
                    "cadd_phred", "revel", "alphamissense", "avi", "gnomad_af",
                    "label"] if c in d.columns]
print(f"\n{len(hits)} variants trip at least one rule\n")
print(hits[cols].head(15).to_string(index=False) if len(hits) else "  (none)")

if len(hits) == 0:
    print("\nNone. That is a legitimate result. Say so in your report and explain why")
    print("this gene's variants are concordant, rather than loosening the rules until")
    print("something fires.")

discordant variants in CYP2C9

flag_cadd_vs_revel     16
flag_revel_vs_am        2
flag_scored_but_vus     0
flag_common_but_plp     3
flag_no_tool_at_all    21
flag_atlas_only         3

41 variants trip at least one rule

      rsid        consequence                          consequence_all aa_change  cadd_phred  revel  alphamissense   avi  gnomad_af label
       NaN frameshift_variant frameshift_variant;splice_region_variant       NaN         NaN    NaN            NaN 12.15   0.003103  B/LB
       NaN frameshift_variant frameshift_variant;splice_region_variant       NaN         NaN    NaN            NaN  7.87   0.003103  B/LB
       NaN frameshift_variant frameshift_variant;splice_region_variant       NaN         NaN    NaN            NaN 11.51   0.003103  B/LB
       NaN frameshift_variant frameshift_variant;splice_region_variant       NaN         NaN    NaN            NaN 10.84   0.003103  B/LB
       NaN frameshift_variant frameshift_variant;splice_region_variant       NaN      